# Library imports

In [1]:
from multiprocessing import Pool
import timeit
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from scipy.stats import nbinom
import os
import xarray as xr
from hurrSim import Simulator, tools 
import hurrSim
from functools import partial
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import datetime

# Set the grid system : 

In [2]:
# Setting the grid system : 
# Longitude :
min_lat = 10 # degree
max_lat = 60 # degree
lat_grids = 5 # degree

# Latitude : 
min_long = 0 # degree
max_long = 110 # degree
long_grids = 5 # degree

# Grid info
grid_width = max_long-min_long
grid_height = max_lat-min_lat
nb_width_cell = int(grid_width/long_grids)
nb_height_cell = int(grid_height/lat_grids)
nb_cells = int((nb_width_cell)*(nb_height_cell))

long_range = np.linspace(min_long,max_long,int(nb_width_cell+1))
lat_range = np.linspace(min_lat,max_lat,int(nb_height_cell+1))

cell_ids = np.flip(np.arange(1,nb_cells+1).reshape(nb_height_cell,nb_width_cell),0)

# Synthesize the grid geometry to a single variable : 
grid = (cell_ids,lat_range,long_range)

# Data imports

In [3]:
data_path = os.path.dirname(hurrSim.__file__) + '\\data\\'

## Import genesis information

In [4]:
# The required information to launch a given number of hurricanes/year :
genesis_sample_param = pd.read_excel(data_path+'genesis_params.xlsx')
display(genesis_sample_param)
n = genesis_sample_param.iloc[0,0]
p = genesis_sample_param.iloc[0,1]

,n,p
0,5.836924,0.340371


In [5]:
# The required information to define where (lat-lon) and how strong (max pressure deficit) do they begin : 
# Get only the 1st reading of the storm
storms_df = pd.read_csv(data_path+'edited_atlantic_storms.csv')

storms_i = storms_df.copy().groupby('id', as_index = False).first()
storms_i = storms_i[['latitude','longitude','pc_final','Int','Vi','theta','id','date']]
storms = storms_i.dropna(axis = 0)
storms = storms[(storms.Vi != 0)] 

# Clear storms which begin outside the grid system :
storms = storms[(storms.latitude < max_lat) & (storms.latitude > min_lat)]
storms = storms[(storms.longitude > -max_long) & (storms.longitude < -min_long)] #careful of negative sign conventions

# Some of HURDAT2 storms are invalid start locations (i.e.: on land) :
invalid_coords  = []
# List and find these invalids records :
for row in storms.iterrows() : 
    lat = row[1][0]
    long = row[1][1]
    land_value = tools.tool_func.is_land(long,lat)
    invalid_coords.append(land_value)
storms['land_check'] = invalid_coords
# Remove from the sampling bank : 
storms = storms[~storms.land_check == True]
# Clear the added column : 
storms = storms.drop(['land_check'],axis = 1)

# Check removed ? 
display_removed = False

if display_removed == True : 
    ds1 = set(map(tuple, storms_i.values))
    ds2 = set(map(tuple, storms.values))
    diff = pd.DataFrame(list(ds1.difference(ds2)))
    display(diff)

# Finaly : 
storms = storms.reset_index(drop = True)
display(storms)

,latitude,longitude,pc_final,Int,Vi,theta,id,date
0,28.0,-94.8,974.265295,0.415090,2.727970,89.859158,AL011851,1851-06-25 00:00:00
1,20.5,-67.1,993.095316,0.226997,4.458666,76.487047,AL011852,1852-08-19 00:00:00
2,26.0,-92.5,990.519179,0.272012,2.314109,89.890407,AL011854,1854-06-25 00:00:00
3,25.0,-83.9,984.965185,0.269447,6.646479,56.902466,AL011856,1856-08-09 00:00:00
4,34.0,-74.5,994.504521,0.323806,3.937790,-48.971990,AL011857,1857-06-30 00:00:00
...,...,...,...,...,...,...,...,...
1781,27.0,-48.0,1010.000000,0.113430,2.768237,-123.807757,AL292005,2005-11-19 12:00:00
1782,14.9,-72.4,1006.000000,0.100048,5.971464,89.845715,AL292020,2020-10-31 18:00:00
1783,31.5,-49.2,993.000000,0.391556,3.548123,81.446086,AL302005,2005-11-29 06:00:00
1784,28.4,-47.5,1010.000000,0.117483,3.170739,-89.833530,AL302020,2020-11-08 12:00:00


In [6]:
# Clear storms which cannot be sampled due to lack of intensity/valid central pressure values :
# This should clear about 60 more storms from the pool.
storms_int_bank = storms[(storms.pc_final < 1013)
                         & (storms.Int != np.nan)]
storms_int_bank = storms_int_bank.reset_index(drop = True)
display(storms_int_bank)

,latitude,longitude,pc_final,Int,Vi,theta,id,date
0,28.0,-94.8,974.265295,0.415090,2.727970,89.859158,AL011851,1851-06-25 00:00:00
1,20.5,-67.1,993.095316,0.226997,4.458666,76.487047,AL011852,1852-08-19 00:00:00
2,26.0,-92.5,990.519179,0.272012,2.314109,89.890407,AL011854,1854-06-25 00:00:00
3,25.0,-83.9,984.965185,0.269447,6.646479,56.902466,AL011856,1856-08-09 00:00:00
4,34.0,-74.5,994.504521,0.323806,3.937790,-48.971990,AL011857,1857-06-30 00:00:00
...,...,...,...,...,...,...,...,...
1724,27.0,-48.0,1010.000000,0.113430,2.768237,-123.807757,AL292005,2005-11-19 12:00:00
1725,14.9,-72.4,1006.000000,0.100048,5.971464,89.845715,AL292020,2020-10-31 18:00:00
1726,31.5,-49.2,993.000000,0.391556,3.548123,81.446086,AL302005,2005-11-29 06:00:00
1727,28.4,-47.5,1010.000000,0.117483,3.170739,-89.833530,AL302020,2020-11-08 12:00:00


## Import track parameters : 

In [30]:
#hurr_param = pd.read_excel(data_path+"grid_data_edited_v3_V_Imin_RH080_allpc.xlsx",index_col = 'grid_ids')
hurr_param = pd.read_excel(data_path+"grid_data_edited_v3_SW_Imin_RH080_allpc.xlsx",index_col = 'grid_ids')

display(hurr_param)

,longitude,latitude,E_alpha_coeffs_ed,E_beta_coeffs_ed,W_alpha_coeffs_ed,W_beta_coeffs_ed,E_kappa_coeffs_ed,W_kappa_coeffs_ed
grid_ids,,,,,,,,
1,107.5,12.5,"[(781.1682535056025, 0.45729229192147614, -0.7...","[(540938.0758666992, 1005.0017541646957, 6143....","[(7.49746074093855, 0.4886387502765501, -0.078...","[(57.74840532336384, -2.1843964177724047, 0.43...","[(-72.12801477705216, 0.923440829175119, -0.13...","[(676.4994673331512, 0.5049011802417638, -0.01..."
2,102.5,12.5,"[(781.1682535056025, 0.45729229192147614, -0.7...","[(540938.0758666992, 1005.0017541646957, 6143....","[(7.49746074093855, 0.4886387502765501, -0.078...","[(57.74840532336384, -2.1843964177724047, 0.43...","[(-72.12801477705216, 0.923440829175119, -0.13...","[(676.4994673331512, 0.5049011802417638, -0.01..."
3,97.5,12.5,"[(781.1682535056025, 0.45729229192147614, -0.7...","[(540938.0758666992, 1005.0017541646957, 6143....","[(1.080523581669695, 0.6875383483909303, -0.02...","[(-103.21421744820691, 6.361451755320463, 0.30...","[(-72.12801477705216, 0.923440829175119, -0.13...","[(676.4994673331512, 0.5049011802417638, -0.01..."
4,92.5,12.5,"[(781.1682535056025, 0.45729229192147614, -0.7...","[(540938.0758666992, 1005.0017541646957, 6143....","[(4.183174132303066, 0.9780265321125654, 0.111...","[(22.31406519614029, 5.649775028162566, 1.2457...","[(-72.12801477705216, 0.923440829175119, -0.13...","[(676.4994673331512, 0.5049011802417638, -0.01..."
5,87.5,12.5,"[(-2.649725156294153, 0.8829161186806935, 0.04...","[(-332.59685801367596, 0.010390189105976333, -...","[(0.7860770944771502, 0.9550862831322746, 0.03...","[(115.77908584475517, -1.3330109702169892, 1.2...","[(-72.12801477705216, 0.923440829175119, -0.13...","[(-89.29140736813861, 1.1110888982051392, -0.0..."
...,...,...,...,...,...,...,...,...
216,22.5,57.5,"[(5.095994105457066, 0.4432276114450815, -0.10...","[(88.06690347758376, -0.6124328368944418, 3.91...","[(-140.81357206776738, 2.074309354473371, 2.78...","[(-25.88659854233265, -8.389334179344587, -24....","[(-38.888589987764135, 13.982872214110103, -10...","[(31.123006426655593, 0.7791365772850796, 0.13..."
217,17.5,57.5,"[(3.295065665115544, 0.8374441487617048, -0.03...","[(514.3999672405334, -11.814746803032875, -12....","[(-140.81357206776738, 2.074309354473371, 2.78...","[(-25.88659854233265, -8.389334179344587, -24....","[(-38.888589987764135, 13.982872214110103, -10...","[(31.123006426655593, 0.7791365772850796, 0.13..."
218,12.5,57.5,"[(1.1095730869865292, 0.9748759911496094, -0.0...","[(115.92331345791763, -3.325830622047423, -2.3...","[(-140.81357206776738, 2.074309354473371, 2.78...","[(-25.88659854233265, -8.389334179344587, -24....","[(1293.9269239529967, -105.2691271645017, 106....","[(31.123006426655593, 0.7791365772850796, 0.13..."


## Bounding storm track speed

In [8]:
# Drop non existing numnbers :
storms_c = storms_df.Vi.dropna()
# Hurdat remains coarse when it comes to long-lat ; 0 speed is somewhat an unlikely bound
storms_c = storms_c[(storms_c !=0)]

# Identify max and min speeds for the simulations :
max_c = storms_c.max()
print(max_c)
min_c = storms_c.min()
print(min_c)
min_c = 0.05 

# Set the speed bounds into a single variable : 
c_bounds = (max_c,min_c)

40.495768312423394
0.3796511892747443


# Import Sea Surface temperature 

In [9]:
# Access a SST value based on grid id or based on lat-long current values
hadsst = xr.open_dataset(data_path+'sliced_HadISST.nc')
display(hadsst)

<xarray.Dataset>
Dimensions:    (time: 1800, latitude: 54, longitude: 114, nv: 2)
Coordinates:
  * time       (time) datetime64[ns] 1870-02-14T23:59:59.340820312 ... 2020-0...
  * latitude   (latitude) float32 61.5 60.5 59.5 58.5 57.5 ... 11.5 10.5 9.5 8.5
  * longitude  (longitude) float32 -111.5 -110.5 -109.5 -108.5 ... -0.5 0.5 1.5
Dimensions without coordinates: nv
Data variables:
    time_bnds  (time, nv) float32 ...
    sst        (time, latitude, longitude) float32 ...
Attributes:
    Title:                      Monthly version of HadISST sea surface temper...
    description:                HadISST 1.1 monthly average sea surface tempe...
    institution:                Met Office Hadley Centre
    source:                     HadISST
    reference:                  Rayner, N. A., Parker, D. E., Horton, E. B., ...
    Conventions:                CF-1.0
    history:                    7/4/2023 converted to netcdf from pp format
    supplementary_information:  Updates and supplementary information will be...
    comment:                    Data restrictions: for academic research use ...

In [10]:
hadsst_pre1870=xr.open_dataset(data_path+'sst_pre_1870.nc')
hadsst_post2020=xr.open_dataset(data_path+'sst_post_2020.nc')
# Quick sanity check : This should output a file with only 12 monthly values available (coordinates variable)
display(hadsst_pre1870)

<xarray.Dataset>
Dimensions:    (month: 12, nv: 2, latitude: 54, longitude: 114)
Coordinates:
  * latitude   (latitude) float32 61.5 60.5 59.5 58.5 57.5 ... 11.5 10.5 9.5 8.5
  * longitude  (longitude) float32 -111.5 -110.5 -109.5 -108.5 ... -0.5 0.5 1.5
  * month      (month) int32 1 2 3 4 5 6 7 8 9 10 11 12
Dimensions without coordinates: nv
Data variables:
    time_bnds  (month, nv) float32 ...
    sst        (month, latitude, longitude) float32 ...
Attributes:
    Title:                      Monthly version of HadISST sea surface temper...
    description:                HadISST 1.1 monthly average sea surface tempe...
    institution:                Met Office Hadley Centre
    source:                     HadISST
    reference:                  Rayner, N. A., Parker, D. E., Horton, E. B., ...
    Conventions:                CF-1.0
    history:                    7/4/2023 converted to netcdf from pp format
    supplementary_information:  Updates and supplementary information will be...
    comment:                    Data restrictions: for academic research use ...

## Import Stratosphere temperature 

In [11]:
air_temp = xr.open_dataset(data_path+'air_mon_temp_sliced.nc')
air_temp_post_2015 = xr.open_dataset(data_path+'air_mon_temp_sliced_2015-2021.nc')
display(air_temp)

<xarray.Dataset>
Dimensions:    (lat: 55, lon: 112, time: 1976, nbnds: 2)
Coordinates:
  * lat        (lat) float32 8.0 9.0 10.0 11.0 12.0 ... 58.0 59.0 60.0 61.0 62.0
  * lon        (lon) float32 248.0 249.0 250.0 251.0 ... 356.0 357.0 358.0 359.0
  * time       (time) datetime64[ns] 1851-05-01 1851-06-01 ... 2015-12-01
Dimensions without coordinates: nbnds
Data variables:
    time_bnds  (time, nbnds) float64 ...
    air        (time, lat, lon) float32 ...
Attributes: (12/24)
    Conventions:               CF-1.2
    title:                     Monthly NOAA/CIRES/DOE 20th Century Reanalysis V3
    comments:                  Data are from \nNOAA/CIRES/DOE 20th Century Re...
    platform:                  Model
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    license:                   These data are available free of charge under ...
    ...                        ...
    citation1:                 Slivinski, L. C, G. P. Compo, J. S. Whitaker, ...
    References:                https://www.psl.noaa.gov/data/gridded/data.20t...
    creator_name:              NOAA/PSL
    institution:               NOAA Physical Sciences Laboratory & CU/CIRES \...
    contact:                   psl.data@noaa.gov
    citation:                  Compo,G.P. <https://www.psl.noaa.gov/people/gi...


## Import storm decay parameters

In [12]:
decay_consts = pd.read_excel(data_path+'decay_constants.xlsx',index_col = 'Location')
display(decay_consts)

,a0,a1,sigma,N
Location,,,,
Gulf coast,0.0413,0.0018,0.0169,26
Florida peninsula coast,0.0225,0.0017,0.0158,13
Mid-Atlantic coast,0.0364,0.0016,0.0161,13
New England coast,0.0034,0.0010,0.0114,6


# Begin simulations

In [28]:
# Simulation parameters :
#---------------------------------------------------------------------------------------
# Generate a directory to store the simulation results - this enables batch processing :
base_dir = r"C:\YOUR\LOCAL\DIRECTORY\TO\STORE\RESULTS"

#---------------------------------------------------------------------------------------
# Number of simulation years desired : 
N =500

In [32]:
now = datetime.datetime.now()
try : 
    job_id = f"{os.environ['COMPUTERNAME']}-{now.year}-{now.month}-{now.day}-{now.hour}h-{now.minute}"
    storage_dir = os.path.join(base_dir,"jobs")
       
    if not os.path.isdir(storage_dir) : 
        os.makedirs(storage_dir)
    
    job_filepath = os.path.join(storage_dir,job_id)
    os.makedirs(job_filepath)
        
except : 
    print('Failed to create the requested storage directory!')

# Start simulating storms :
#----------------------------------------------------------------------------------------
# Sample the number of hurricanes/year :
sample = nbinom.rvs(n=n, p=p, size=N)

# Genesis count simulated through an MLE
#sample_nb = 2
#sample = pd.read_csv('sampled_matlab.csv', header = None).to_numpy()
#sample= sample[:,0][((sample_nb-1)*N):(sample_nb*N)]
#sample.shape
#-----------------------------------------------------------------------------------------
#  Quick look at the number of hurricanes/sampled years for the first 15 : 
print('Here are the first values from the sample years : \n',sample[:15])
print(f'This sample has the following characteristics : Minimum = {np.min(sample)}; Maximum = {np.max(sample)} ;' 
      f' Average = {np.mean(sample):.2f} (hurricanes / year)')    
#-----------------------------------------------------------------------------------------
# Begin dispatching storm simulations : 

# Storage variables :
hurr_ids = []
lat_rec = []
long_rec = []
cell_rec = []
c_rec = []
theta_rec = []
land_rec = []
tag_rec = []
Pc_rec = []
RMW_rec = []
Int_rec = []
Holland_B_rec = []
delta_p_rec = []
sst_failed_rec = []
time_rec = []


print('Beginning Simulations...')
start = timeit.default_timer()

# Variables to include within the "multiprocessing call" : 
Sim_args = partial(Simulator.Simulator_A,   # the function
        sample,                             # Variables required by the function
        storms,
        storms_int_bank,
        grid,
        hurr_param,
        c_bounds,
        air_temp_post_2015,
        air_temp,
        hadsst_pre1870,
        hadsst_post2020,
        hadsst,
        decay_consts
                  )

# Launch simulations : 
if __name__ == "__main__" :
    try: 
        with Pool(os.cpu_count()-1) as pool :
            with tqdm(total = N, desc='Number of years successfully processed : ') as pbar1 :
                for processed in pool.imap(Sim_args,range(N)) :
                    # Fill the foreground recorders : 
                    hurr_ids.extend(processed[0])
                    time_rec.extend(processed[1])
                    lat_rec.extend(processed[2])
                    long_rec.extend(processed[3])
                    cell_rec.extend(processed[4])
                    c_rec.extend(processed[5])
                    theta_rec.extend(processed[6])
                    sst_failed_rec.extend(processed[7])
                    land_rec.extend(processed[8])
                    tag_rec.extend(processed[9])
                    Pc_rec.extend(processed[10])                    
                    Int_rec.extend(processed[11])
                    RMW_rec.extend(processed[12])
                    Holland_B_rec.extend(processed[13])
                    # Update the progress bar : 
                    pbar1.update()
    except : 
        print('Warning, an iteration failed in this sample.')
end = timeit.default_timer()   
print(f'Simulations finished! Elapsed time : {end-start:.2f} seconds')
#-----------------------------------------------------------------------------------------
# Save everything into the directory : 
print('Saving simulation outputs...')

# Build a dataframe off the results :
# Column names : 
cols = ['Hurr_ids','time','latitude','longitude','cell_id','c_i','theta_i','land status','invalid sst_status', 'tag','pc','Int','RMW','Holland B']

# Setting everything in the form of a dataframe :
Hurr_sim_db = pd.DataFrame([hurr_ids,time_rec,lat_rec,long_rec,cell_rec,c_rec,theta_rec,land_rec,sst_failed_rec,tag_rec,Pc_rec,Int_rec,RMW_rec,Holland_B_rec]).T
Hurr_sim_db.columns = cols
#print('The amount of simulated storms is :',len(pd.unique(Hurr_sim_db['Hurr_ids'])))


# Save the dataframe : 
# Note : CSV format is prefered over Excel, as datetimes start before 1900s
df_path = os.path.join(base_dir,"jobs",job_id,"Simulations.csv")
Hurr_sim_db.to_csv(df_path)

# A pickle file is also created for easy access in post-processing : 
pickle_path = os.path.join(base_dir,"jobs",job_id,"Pickled_simulations.pickle")
Hurr_sim_db.to_pickle(pickle_path)

# Save a copy of the random number simulator producing the amount of storms :
sample_path = os.path.join(base_dir,"jobs",job_id,"sample")
np.save(sample_path,sample)

print('Done!')

display(Hurr_sim_db.head(10))

Here are the first values from the sample years : 
 [15  8 10  7 17 10 11  9  5 16  8 11  9 26 14]
This sample has the following characteristics : Minimum = 1; Maximum = 32 ; Average = 11.42 (hurricanes / year)
Beginning Simulations...


Number of years successfully processed :   0%|          | 0/500 [00:00<?, ?it/s]

Warning, an iteration failed in this sample.
Simulations finished! Elapsed time : 22.73 seconds
Saving simulation outputs...
Done!


,Hurr_ids,time,latitude,longitude,cell_id,c_i,theta_i,land status,invalid sst_status,tag,pc,Int,RMW,Holland B
